# EternityQuant — Kaggle ML 训练

在 Kaggle 的免费 GPU（T4/P100）上训练 EternityQuant 的量化选股模型。

## 支持算法
- **LightGBM** (CPU) — 基线模型，qlib Alpha158 特征
- **MLP** (CUDA) — 自写全连接网络，158→512→256→128→1
- **GRU** (CUDA) — 自写时序模型，6×26 时序重塑，**量化选股最佳**
- **LSTM** (CUDA) — 与 GRU 同架构，3 门替代 2 门

## 输出
- 训练好的模型文件（`.pkl`）→ Kaggle Output 中下载
- 训练指标（IC、ICIR、Rank IC）

---

## 1. 环境准备

Kaggle 预装 PyTorch CUDA，只需安装 qlib 和 lightgbm。

In [ ]:
# @title 安装依赖
import sys, subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs))

_pip("numpy", "pandas", "typer", "python-dotenv")

# qlib — Kaggle 上编译较慢，预编译 wheel 优先
_pip("pyqlib>=0.9")

_pip("lightgbm>=4.0")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("依赖安装完成 ✓")

In [ ]:
# @title 克隆 EternityQuant
import os
REPO_URL = "https://github.com/your-username/EternityQuant.git"  # TODO: 替换为你的仓库地址
PROJECT_DIR = "/kaggle/working/EternityQuant"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
!pip install -e . --quiet
print(f"工作目录: {os.getcwd()}")

## 2. 准备 qlib 数据

### 方案 A：使用 Kaggle Dataset（推荐）
1. 在 Kaggle 上创建一个 Dataset，上传 `cn_data.tar.gz`（qlib 数据包）
2. 在当前 Notebook 的右侧 "Add Data" 中搜索并添加该 Dataset
3. 数据会自动挂载到 `/kaggle/input/` 目录

### 方案 B：在线拉取（慢）
Kaggle 可以访问 baostock，直接运行 `eq ml update-data` 拉取。

In [ ]:
# @title 设置 qlib 数据路径
import os, shutil

QLIB_TARGET = "/kaggle/working/.qlib_data/cn_data"

# 方案 A：从 Kaggle Dataset 中复制
KAGGLE_INPUT = "/kaggle/input/eternityquant-qlib-data/cn_data"  # TODO: 替换为你的 Dataset 路径
if os.path.exists(KAGGLE_INPUT):
    os.makedirs("/kaggle/working/.qlib_data", exist_ok=True)
    if not os.path.exists(QLIB_TARGET):
        shutil.copytree(KAGGLE_INPUT, QLIB_TARGET)
        print(f"已从 Kaggle Dataset 复制 qlib 数据: {KAGGLE_INPUT}")
    else:
        print(f"qlib 数据已存在: {QLIB_TARGET}")
else:
    print(f"Kaggle Dataset 未找到，将使用在线拉取")

# 验证
if os.path.exists(QLIB_TARGET):
    !ls -la {QLIB_TARGET}/calendars/ 2>/dev/null || echo "日历文件未找到"
else:
    print("qlib 数据目录不存在，需要在线拉取")

In [ ]:
# @title (方案 B) 在线拉取 qlib 数据
import os
QLIB_TARGET = "/kaggle/working/.qlib_data/cn_data"
if not os.path.exists(QLIB_TARGET):
    !mkdir -p /kaggle/working/.qlib_data
    !cd /kaggle/working/EternityQuant && python -c "
from eq.strategy.factors.ml_data_updater import update_qlib_data
result = update_qlib_data(start='2015-01-01', universe='csi300', workers=8, verbose=True)
print(f'数据更新完成: {result}')
"
else:
    print(f"qlib 数据已存在: {QLIB_TARGET}")

## 3. 训练模型

### 3.1 LightGBM (CPU 基线)

In [ ]:
# @title 训练 LightGBM
import sys, os
os.chdir("/kaggle/working/EternityQuant")
sys.path.insert(0, ".")

from eq.strategy.factors.ml_workflow import train

result = train(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="lightgbm",
    device="cpu",
    name="kaggle_lgbm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.2 MLP (CUDA)

In [ ]:
# @title 训练 MLP
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="mlp",
    device="cuda",
    name="kaggle_mlp_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.3 GRU (CUDA) — 推荐，量化选股最佳

In [ ]:
# @title 训练 GRU
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="gru",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=4000,
    name="kaggle_gru_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.4 LSTM (CUDA)

In [ ]:
# @title 训练 LSTM
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="lstm",
    device="cuda",
    hidden_size=128,
    num_layers=2,
    batch_size=4000,
    name="kaggle_lstm_csi300_h5",
)
print(f"\n训练完成:")
print(f"  Model ID: {result['model_id']}")
print(f"  IC: {result['metrics']['ic']:.4f}")
print(f"  模型文件: {result['model_path']}")

### 3.5 DeepLOB (CUDA) — CNN+BiLSTM+Attention

[Zhang et al., 2019] 针对金融微观结构设计的专用架构。

In [ ]:
# @title 训练 DeepLOB
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="deeplob",
    device="cuda",
    optimizer="adamw",
    loss="sharpe",
    batch=512,
    name="kaggle_deeplob_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.6 TFT (CUDA) — Temporal Fusion Transformer

[Lim et al., 2019] Google 多时间跨度预测模型。

In [ ]:
# @title 训练 TFT
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss="sharpe",
    hidden=256,
    heads=4,
    batch=256,
    name="kaggle_tft_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.7 高级优化器对比

SAM（平坦极小值）/ Lookahead（双权重稳定）/ Lion（符号梯度）

In [ ]:
# @title GRU + SAM 优化器 + 可微夏普比率
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="sam",
    loss="sharpe",
    name="kaggle_gru_sam_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

In [ ]:
# @title GRU + Lion 优化器
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="gru",
    device="cuda",
    optimizer="lion",
    loss="sharpe",
    name="kaggle_gru_lion_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.8 对抗训练 + 特征正交化

In [ ]:
# @title TFT + 对抗训练 + 特征正交化
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import train_torch

result = train_torch(
    universe="csi300",
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    horizon=5,
    algo="tft",
    device="cuda",
    optimizer="adamw",
    loss="sharpe",
    adversarial=True,
    orthogonalize=True,
    name="kaggle_tft_adv_orth_csi300_h5",
)
print(f"\n训练完成: IC={result['metrics']['ic']:+.4f}")

### 3.9 超参搜索

In [ ]:
# @title GRU 超参网格搜索
import sys, os
os.chdir("/kaggle/working/EternityQuant")

from eq.strategy.factors.ml_workflow import search_lstm

results = search_lstm(
    universe="csi300",
    horizon=5,
    train_start="2015-01-01",
    train_end="2020-08-31",
    valid_start="2020-09-01",
    valid_end="2020-09-25",
    device="cuda",
    fast=True,
    auto_train=True,
    algo="gru",
)

print(f"\n搜索完成 {len(results)} 组合")
for i, r in enumerate(results[:3]):
    print(f"  #{i+1}: hidden={r['hidden_size']} layers={r['num_layers']} "
          f"lr={r['lr']} batch={r['batch_size']}  IC={r['ic']:+.4f}")

## 4. 导出模型

Kaggle 的输出目录 `/kaggle/working/` 中的文件会在 Notebook 运行结束后打包为 Output。
模型文件在 `/kaggle/working/EternityQuant/.eternityquant/ml_models/` 下。

In [ ]:
# @title 列出所有训练好的模型
import os
models_dir = "/kaggle/working/EternityQuant/.eternityquant/ml_models"
if os.path.exists(models_dir):
    for f in sorted(os.listdir(models_dir)):
        size = os.path.getsize(os.path.join(models_dir, f))
        print(f"  {f:45s}  {size/1024:.1f} KB")
else:
    print("未找到模型文件")

In [ ]:
# @title 复制模型到 Kaggle Output 目录
import os, shutil
models_dir = "/kaggle/working/EternityQuant/.eternityquant/ml_models"
output_dir = "/kaggle/working/models"

if os.path.exists(models_dir) and os.listdir(models_dir):
    shutil.copytree(models_dir, output_dir, dirs_exist_ok=True)
    print(f"模型已复制到 {output_dir}")
    !ls -la {output_dir}/
else:
    print("没有模型文件")

## 5. 回本地导入模型

从 Kaggle Output 下载模型文件后：

```bash
# 解压模型文件
tar xzf models.tar.gz -C ~/.eternityquant/ml_models/

# 登记模型
eq ml register --name "kaggle_gru_csi300_h5" --universe csi300 \
    --features "Alpha158" --algo gru --horizon 5 \
    --train-period "2015-01-01~2020-08-31" \
    --model-path ~/.eternityquant/ml_models/torch_gru_csi300_5d.pkl \
    --metrics '{"ic": 0.15}' \
    --notes "Kaggle GPU 训练"

# 激活模型
eq ml activate <model_id>

# 批量预测
eq ml predict-batch <model_id> --top 10
```

---

## 附：Kaggle GPU 配置信息

In [ ]:
# @title GPU 信息
import torch, os
print(f"CPU: {os.cpu_count()} 核")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU: 不可用")